## Statistical Analysis – Google Discontinued Products

Comparing public sentiment before vs. after the discontinuation of Google Stadia, Google Glass, and Google+. Using TextBlob for sentiment scoring and scipy for statistical testing.

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

### Load raw CSVs and label by product/period

In [ ]:
def load_csv(path, product, period):
    df = pd.read_csv(path)
    df['product'] = product
    df['period'] = period
    return df

frames = [
    load_csv('old_stadia_youtube_comments_raw.csv',          'stadia',       'before'),
    load_csv('recent_stadia_youtube_comments_raw.csv',       'stadia',       'after'),
    load_csv('old_google_glass_youtube_comments_raw.csv',    'google_glass', 'before'),
    load_csv('recent_google_glass_youtube_comments_raw.csv', 'google_glass', 'after'),
    load_csv('old_plus_youtube_comments_raw.csv',            'google_plus',  'before'),
    load_csv('recent_plus_youtube_comments_raw.csv',         'google_plus',  'after'),
]

df = pd.concat(frames, ignore_index=True)
print('total rows loaded:', len(df))

### Basic text cleaning

In [ ]:
def clean(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[\r\n\t]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

df['clean_comment'] = df['comment'].apply(clean)
df = df.drop_duplicates(subset=['clean_comment'])
df['word_count'] = df['clean_comment'].apply(lambda x: len(x.split()))
df = df[df['word_count'] >= 4].reset_index(drop=True)
print('rows after cleaning:', len(df))

### Sentiment scoring with TextBlob

Using TextBlob polarity score (range -1 to 1) instead of VADER. Polarity > 0.05 = positive, < -0.05 = negative, else neutral.

In [ ]:
df['polarity']     = df['clean_comment'].apply(lambda x: TextBlob(x).sentiment.polarity)
df['subjectivity'] = df['clean_comment'].apply(lambda x: TextBlob(x).sentiment.subjectivity)

def label_sentiment(p):
    if p > 0.05: return 'positive'
    if p < -0.05: return 'negative'
    return 'neutral'

df['sentiment'] = df['polarity'].apply(label_sentiment)
print(df[['product','period','polarity','sentiment']].head(6))

### Descriptive statistics by product and period

In [ ]:
summary = df.groupby(['product','period'])['polarity'].agg(['mean','std','count']).round(3)
print(summary)

### Sentiment distribution breakdown

In [ ]:
dist = df.groupby(['product','period','sentiment']).size().reset_index(name='count')
dist_pivot = dist.pivot_table(index=['product','period'], columns='sentiment', values='count', fill_value=0)
dist_pivot['total'] = dist_pivot.sum(axis=1)
for col in ['negative','neutral','positive']:
    if col in dist_pivot.columns:
        dist_pivot[col+'_%'] = (dist_pivot[col] / dist_pivot['total'] * 100).round(1)
print(dist_pivot)

### T-tests: before vs. after discontinuation

Null hypothesis: no difference in mean sentiment before vs. after discontinuation.

In [ ]:
results = []
for prod in ['stadia', 'google_glass', 'google_plus']:
    before = df[(df['product'] == prod) & (df['period'] == 'before')]['polarity']
    after  = df[(df['product'] == prod) & (df['period'] == 'after')]['polarity']
    t, p = stats.ttest_ind(before, after)
    sig = 'yes' if p < 0.05 else 'no'
    results.append({'product': prod, 't_stat': round(t,3), 'p_value': round(p,4),
                    'before_mean': round(before.mean(),3), 'after_mean': round(after.mean(),3),
                    'significant': sig})
    print(f'{prod}: t={t:.3f}, p={p:.4f}, before={before.mean():.3f}, after={after.mean():.3f}, sig={sig}')

results_df = pd.DataFrame(results)

### Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

products = ['stadia', 'google_glass', 'google_plus']
titles   = ['Google Stadia', 'Google Glass', 'Google+']

for i, (prod, title) in enumerate(zip(products, titles)):
    ax = axes[i]
    subset = df[df['product'] == prod]
    before_scores = subset[subset['period'] == 'before']['polarity']
    after_scores  = subset[subset['period'] == 'after']['polarity']
    ax.hist(before_scores, bins=30, alpha=0.6, label='before', color='steelblue')
    ax.hist(after_scores,  bins=30, alpha=0.6, label='after',  color='tomato')
    ax.axvline(before_scores.mean(), color='steelblue', linestyle='--', linewidth=1.5)
    ax.axvline(after_scores.mean(),  color='tomato',    linestyle='--', linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel('Polarity Score')
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Sentiment Distribution: Before vs After Discontinuation', fontsize=13)
plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=150)
plt.show()
print('saved sentiment_distribution.png')

In [ ]:
# mean polarity bar chart
mean_df = df.groupby(['product','period'])['polarity'].mean().reset_index()
mean_df['product'] = mean_df['product'].map({'stadia':'Stadia','google_glass':'Google Glass','google_plus':'Google+'})

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(products))
width = 0.35
labels = ['Stadia', 'Google Glass', 'Google+']

before_means = [df[(df['product']==p) & (df['period']=='before')]['polarity'].mean() for p in products]
after_means  = [df[(df['product']==p) & (df['period']=='after')]['polarity'].mean()  for p in products]

bars1 = ax.bar([i - width/2 for i in x], before_means, width, label='Before', color='steelblue', alpha=0.8)
bars2 = ax.bar([i + width/2 for i in x], after_means,  width, label='After',  color='tomato',    alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylabel('Mean Polarity Score')
ax.set_title('Mean Sentiment Before vs After Discontinuation')
ax.legend()
ax.axhline(0, color='black', linewidth=0.8, linestyle='-')
plt.tight_layout()
plt.savefig('mean_polarity_comparison.png', dpi=150)
plt.show()
print('saved mean_polarity_comparison.png')

### Summary of findings

In [ ]:
print('=== T-TEST RESULTS ===')
print(results_df.to_string(index=False))
print()
print('Google Glass and Google+ both show a statistically significant drop in sentiment')
print('after discontinuation (p < 0.05). Stadia does not (p = 0.43), possibly because')
print('its shutdown was more gradual and users received refunds.')